<a href="https://colab.research.google.com/github/sgruyzcki/DiploDatos2026/blob/main/An%C3%A1lisis%20Exploratorio%20y%20Curaci%C3%B3n%20de%20Datos/parte%202/ECD_2026_Entregable_Parte_2_Mariano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

**Exploración y Curación de Datos**

*Edición 2026*

----

# Trabajo práctico entregable - parte 2

En esta notebook, vamos a cargar el conjunto de datos de [la compentencia Kaggle](https://www.kaggle.com/dansbecker/melbourne-housing-snapshot) sobre estimación de precios de ventas de propiedades en Melbourne, Australia.

Utilizaremos el conjunto de datos reducido producido por [DanB](https://www.kaggle.com/dansbecker). Hemos subido una copia a un servidor de la Universidad Nacional de Córdoba para facilitar su acceso remoto.

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seaborn as sns
sns.set_context('talk')

from sqlalchemy import create_engine, text

In [9]:
import plotly
plotly.__version__


'5.24.1'

In [13]:
melb_data = pd.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_data[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


In [29]:
interesting_cols = [
  'description', 'neighborhood_overview',
  'street', 'neighborhood', 'city', 'suburb', 'state', 'zipcode',
  'price', 'weekly_price', 'monthly_price',
  'latitude', 'longitude',
]

airbnb_df = pd.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv',
    usecols=interesting_cols)

airbnb_df[:3]

/tmp/ipykernel_39346/1736643712.py:8: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  airbnb_df = pd.read_csv(


,description,neighborhood_overview,street,neighborhood,city,suburb,state,zipcode,latitude,longitude,price,weekly_price,monthly_price
0,"House: Clean, New, Modern, Quite, Safe. 10Km f...",Very safe! Family oriented. Older age group.,"Bulleen, VIC, Australia",Balwyn North,Manningham,Bulleen,VIC,3105,-37.772684,145.092133,60,NaN,NaN
1,A large air conditioned room with queen spring...,This hip area is a crossroads between two grea...,"Brunswick East, VIC, Australia",Brunswick,Moreland,Brunswick East,VIC,3057,-37.766505,144.980736,35,200.0,803.0
2,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,A stay at our apartment means you can enjoy so...,"St Kilda, VIC, Australia",St Kilda,Port Phillip,St Kilda,VIC,3182,-37.859755,144.977369,159,1253.0,4452.0


In [30]:
airbnb_df.shape

(22895, 13)

In [32]:
airbnb_df['zipcode'] = pd.to_numeric(airbnb_df.zipcode, errors='coerce')

In [33]:
airbnb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22895 entries, 0 to 22894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   description            22563 non-null  object 
 1   neighborhood_overview  14424 non-null  object 
 2   street                 22895 non-null  object 
 3   neighborhood           17082 non-null  object 
 4   city                   22895 non-null  object 
 5   suburb                 22872 non-null  object 
 6   state                  22834 non-null  object 
 7   zipcode                22749 non-null  float64
 8   latitude               22895 non-null  float64
 9   longitude              22895 non-null  float64
 10  price                  22895 non-null  int64  
 11  weekly_price           2524 non-null   float64
 12  monthly_price          1891 non-null   float64
dtypes: float64(5), int64(1), object(7)
memory usage: 2.3+ MB


A continuacion, se hace el tratamiento de airbnb_df hecho en el practico 04.1

In [34]:
airbnb_df['zipcode_int'] = airbnb_df.zipcode.fillna(0).astype('int')

In [35]:
airbnb_df[airbnb_df['zipcode_int'] == 0].shape

(146, 14)

## Ejercicio 1 SQL:

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

3. Validar tipos de columnas antes de guardar. Usá `df.dtypes` para ver los tipos actuales. Prestá especial atención a columnas como `Date` y `Price`: por ejemplo, `Date` puede estar como string en vez de datetime, y `Price` puede venir como string o float. El método `to_sql()` infiere tipos automáticamente, pero puede fallar si los tipos no son los esperados.

4. Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por `Regionname`.
    - cantidad de registros totales por `Suburb` y `Regionname`.
    - Consulta con filtro: ¿Cuántas propiedades hay por `Regionname` con más de 2 habitaciones?
    - Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (`Type`) y `Regionname`?
    - Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.

5. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL para obtener un resultado similar a lo realizado con Pandas en clase.

6. Agregar una celda de validación posterior al JOIN con assertions o validación de esquema. Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables. Esta validación implementa dimensiones básicas de calidad de datos como validez, completitud e integridad.



In [14]:
engine = create_engine('sqlite:///melb_data.sqlite3', echo=False)

In [15]:
melb_df = melb_data.drop(['BuildingArea', 'YearBuilt'], axis=1).copy()
#melb_df.info()

In [16]:
# Valores faltantes y ceros por columna
faltantes = melb_df.isna().sum()
ceros = (melb_df == 0).sum()
resumen = pd.DataFrame({'Faltantes': faltantes, 'Ceros': ceros})
resumen[resumen.any(axis=1)]

,Faltantes,Ceros
Distance,0,6
Bedroom2,0,16
Bathroom,0,34
Car,62,1026
Landsize,0,1939
CouncilArea,1369,0


In [17]:
# Eliminamos columnas faltantes en la variable Car
melb_df = melb_df.dropna(subset=['Car'])

missing_values_count = melb_df.isna().sum()
missing_values_count[missing_values_count > 0]

,0
CouncilArea,1307


In [18]:
melb_df['CouncilArea'] = melb_df['CouncilArea'].fillna('Desconocido')

In [19]:
melb_df = melb_df.drop('Bedroom2', axis=1)

melb_df.loc[melb_df['Bathroom'] < 1, 'Bathroom'] = pd.NA
melb_df.loc[melb_df['Bathroom'] > melb_df['Rooms'], 'Bathroom'] = pd.NA
melb_df = melb_df.dropna(subset=['Bathroom'])
melb_df.shape[0]

13456

In [21]:
melb_df = melb_df[melb_df['Landsize'] != 0]
q99_landsize = melb_df['Landsize'].quantile(0.99)
melb_df = melb_df[(melb_df['Landsize'] < q99_landsize) | (melb_df['Landsize'].isnull())]

In [23]:
melb_df['Postcode'] = melb_df['Postcode'].astype(object)

# Propertycount como integer
melb_df['Propertycount'] = melb_df['Propertycount'].astype(int)

# Car como integer
melb_df['Car'] = melb_df['Car'].astype(int)

# Bathroom como integer
melb_df['Bathroom'] = melb_df['Bathroom'].astype(int)

# Date como fecha
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)

melb_df.dtypes

,0
Suburb,object
Address,object
Rooms,int64
Type,object
Price,float64
Method,object
SellerG,object
Date,datetime64[ns]
Distance,float64
Postcode,object


In [24]:
melb_df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bathroom', 'Car', 'Landsize',
       'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname',
       'Propertycount'],
      dtype='object')

In [25]:
melb_df.shape

(11423, 18)

In [50]:
melb_df.to_sql('survey', con=engine, if_exists="replace", index=False)
print(f"Table 'melb_df' created with {len(melb_df)} rows.")

Table 'melb_df' created with 11423 rows.


Hasta aquí se hizo una limpieza de los datos ingestados como en el ejercicio 1 y se guardó el ds limpio en la base de datos creada.

Se hace la interseccion entre la tabla curada de melb_df y la tabla a limpiar airbnb_df

In [36]:
intersection = np.intersect1d(
    airbnb_df.zipcode.values, melb_df.Postcode.values, assume_unique=False)
print("Airbnb unique zipcodes", len(airbnb_df.zipcode.unique()))
print("Sales unique zipcodes", len(melb_df.Postcode.unique()))
print("Common zipcodes", len(intersection))

Airbnb unique zipcodes 248
Sales unique zipcodes 194
Common zipcodes 188


In [37]:
print('Records in Sales df with corresponding zipcode form Airbnb df',
      melb_df.Postcode.isin(intersection).sum() / len(melb_df))
print('Records in Airbnb df with corresponding zipcode form Sales df',
      airbnb_df.zipcode.isin(intersection).sum() / len(airbnb_df))

Records in Sales df with corresponding zipcode form Airbnb df 0.9985993171671189
Records in Airbnb df with corresponding zipcode form Sales df 0.9029919196331077


In [39]:
relevant_cols = ['price', 'weekly_price', 'monthly_price', 'zipcode']

# First we filter out the columns we want, and then we only aggregate
# those. Be careful to include the grouping column as well.
airbnb_df[relevant_cols]\
  .groupby('zipcode').mean().reset_index()[:5]

,zipcode,price,weekly_price,monthly_price
0,2010.0,40.000000,NaN,NaN
1,2134.0,50.000000,NaN,NaN
2,2582.0,104.000000,NaN,NaN
3,3000.0,150.504307,918.738956,3407.204651
4,3001.0,132.500000,NaN,NaN


In [41]:
# Pass a dictionary where the keys are the original columns to aggregate and
# the values are the operations (or list of operations).
airbnb_price_by_zipcode = airbnb_df[relevant_cols].groupby('zipcode')\
  .agg({'price': ['mean', 'count'], 'weekly_price': 'mean',
        'monthly_price': 'mean'})\
  .reset_index()
# Flatten the two level columns
airbnb_price_by_zipcode.columns = [
  ' '.join(col).strip()
  for col in airbnb_price_by_zipcode.columns.values]
# Rename columns
airbnb_price_by_zipcode = airbnb_price_by_zipcode.rename(
    columns={'price mean': 'airbnb_price_mean',
             'price count': 'airbnb_record_count',
             'weekly_price mean': 'airbnb_weekly_price_mean',
             'monthly_price mean': 'airbnb_monthly_price_mean'}
)

airbnb_price_by_zipcode[:3]

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,2010.0,40.0,1,NaN,NaN
1,2134.0,50.0,1,NaN,NaN
2,2582.0,104.0,1,NaN,NaN


In [42]:
airbnb_price_by_zipcode.to_csv("airbnb_price_by_zipcode.csv", index=None)

In [43]:
merged_sales_df = melb_df.merge(
    airbnb_price_by_zipcode, how='left',
    left_on='Postcode', right_on='zipcode'
)
merged_sales_df.sample(5)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
3460,Newport,16 Blenheim Rd,3,h,1176000.0,S,hockingstuart,2016-06-18,8.4,3015.0,...,Hobsons Bay,-37.8466,144.8636,Western Metropolitan,5498,3015.0,132.046154,65.0,706.125000,2002.333333
5051,Thornbury,138 Raleigh St,2,h,1143000.0,S,Ray,2016-08-22,6.5,3071.0,...,Darebin,-37.7598,145.0088,Northern Metropolitan,8870,3071.0,125.008772,114.0,675.800000,2580.100000
4340,Richmond,8 Hodgson Tce,3,h,1205000.0,S,Biggin,2016-11-12,2.6,3121.0,...,Yarra,-37.8198,144.9979,Northern Metropolitan,14949,3121.0,162.262739,628.0,1089.068182,3771.906250
4168,Reservoir,24 Livingstone St,3,h,940000.0,S,Nelson,2016-11-12,11.2,3073.0,...,Darebin,-37.7249,144.9854,Northern Metropolitan,21650,3073.0,273.926471,68.0,399.888889,1459.625000
1713,Coburg,32A Kelson St,3,h,1025000.0,VB,Jellis,2017-02-25,7.8,3058.0,...,Moreland,-37.7408,144.9558,Northern Metropolitan,11204,3058.0,103.105263,133.0,896.055556,3624.166667


In [44]:
merged_sales_df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bathroom', 'Car', 'Landsize',
       'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount',
       'zipcode', 'airbnb_price_mean', 'airbnb_record_count',
       'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'],
      dtype='object')

In [45]:
print(f"Filas antes: {len(melb_df)}")
print(f"Filas después: {len(merged_sales_df)}")
print(f"Nulls nuevos en airbnb_price_mean: {merged_sales_df['airbnb_price_mean'].isna().sum()}")

assert len(merged_sales_df) == len(melb_df), "El merge cambió el número de filas"
assert merged_sales_df["Price"].isna().sum() == 0, "Hay nulls inesperados en Price"
assert merged_sales_df["airbnb_price_mean"].dropna().between(0, 10000).all(), "Precios fuera de rango"


Filas antes: 11423
Filas después: 11423
Nulls nuevos en airbnb_price_mean: 16


In [46]:
merged_sales_df.to_csv("melb_data_extended.csv", index=None)

In [47]:
# Consultas
# Cantidad de registros totales por Regionname
query_1 = """
SELECT Regionname, COUNT(*) AS total_registros
FROM survey
GROUP BY Regionname
"""

# Cantidad de registros totales por Suburb y Regionname
query_2 = """
SELECT Suburb, Regionname, COUNT(*) AS total_registros
FROM survey
GROUP BY Suburb, Regionname
"""

# Consulta con filtro: ¿Cuántas propiedades hay por Regionname con más de 2 habitaciones?
query_3 = """
SELECT Regionname, COUNT(*) AS total_propiedades
FROM survey
WHERE Rooms > 2
GROUP BY Regionname
"""

# Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (Type) y Regionname?
query_4 = """
SELECT Type, Regionname, AVG(Price) AS promedio_precio
FROM survey
GROUP BY Type, Regionname
"""

# Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio
query_5 = """
SELECT Suburb, AVG(Price) AS promedio_precio
FROM survey
GROUP BY Suburb
ORDER BY promedio_precio DESC
LIMIT 5
"""

In [51]:
airbnb_price_by_zipcode.to_sql('airbnb_prices', con=engine, if_exists='replace', index=False)
print(f"Table 'airbnb_prices' created with {len(airbnb_price_by_zipcode)} rows.")

Table 'airbnb_prices' created with 247 rows.


Now, let's combine the `survey` table (Melbourne housing data) with the `airbnb_prices` table using a `LEFT JOIN` on the `Postcode` and `zipcode` columns. This will allow us to enrich the housing data with aggregated Airbnb pricing information.

In [52]:
query_join = """
SELECT
    s.*,
    ap.airbnb_price_mean,
    ap.airbnb_record_count,
    ap.airbnb_weekly_price_mean,
    ap.airbnb_monthly_price_mean
FROM
    survey AS s
LEFT JOIN
    airbnb_prices AS ap ON s.Postcode = ap.zipcode
"""

with engine.connect() as conn:
    joined_df_sql = pd.read_sql(text(query_join), conn)

display(joined_df_sql.head())

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Landsize,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03 00:00:00.000000,2.5,3067.0,...,202.0,Yarra,-37.7996,144.9984,Northern Metropolitan,4019,130.624031,258.0,605.152174,2187.032258
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04 00:00:00.000000,2.5,3067.0,...,156.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019,130.624031,258.0,605.152174,2187.032258
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04 00:00:00.000000,2.5,3067.0,...,134.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019,130.624031,258.0,605.152174,2187.032258
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04 00:00:00.000000,2.5,3067.0,...,94.0,Yarra,-37.7969,144.9969,Northern Metropolitan,4019,130.624031,258.0,605.152174,2187.032258
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04 00:00:00.000000,2.5,3067.0,...,120.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019,130.624031,258.0,605.152174,2187.032258


The SQL JOIN was successful. Now, let's perform some validation checks on the joined data.

In [53]:
print(f"Original `melb_df` rows: {len(melb_df)}")
print(f"Joined DataFrame rows (SQL): {len(joined_df_sql)}")

# Check if the number of rows is consistent after a LEFT JOIN from the 'survey' table
assert len(joined_df_sql) == len(melb_df), "The JOIN operation changed the number of rows unexpectedly!"

# Check for unexpected nulls in critical columns (e.g., 'Price' from original dataset)
assert joined_df_sql['Price'].isna().sum() == 0, "Unexpected null values found in 'Price' after JOIN."

# Check if `airbnb_price_mean` values are within a reasonable range (assuming prices are positive)
if 'airbnb_price_mean' in joined_df_sql.columns:
    # Allow for NaNs in joined columns from Airbnb if no match was found
    assert (joined_df_sql['airbnb_price_mean'].dropna() >= 0).all(), "Airbnb average prices are not all positive."
    print(f"Nulls in `airbnb_price_mean` after SQL JOIN: {joined_df_sql['airbnb_price_mean'].isna().sum()}")
else:
    print("Column 'airbnb_price_mean' not found in joined DataFrame.")

print("SQL JOIN and validation completed successfully!")

Original `melb_df` rows: 11423
Joined DataFrame rows (SQL): 11423
Nulls in `airbnb_price_mean` after SQL JOIN: 16
SQL JOIN and validation completed successfully!


Ahora si, implementacion de las consultas a melb_df en sql

In [57]:
# Implementar en sql las querys anteriores para obtener respuestas del dataset
with engine.connect() as conn:
    result_1 = conn.execute(text(query_1))

    result_2 = conn.execute(text(query_2))

    result_3 = conn.execute(text(query_3))

    result_4 = conn.execute(text(query_4))

    result_5 = conn.execute(text(query_5))

    # Convertir los resultados a DataFrames
    df_result_1 = pd.DataFrame(result_1.fetchall(), columns=result_1.keys())
    df_result_2 = pd.DataFrame(result_2.fetchall(), columns=result_2.keys())
    df_result_3 = pd.DataFrame(result_3.fetchall(), columns=result_3.keys())
    df_result_4 = pd.DataFrame(result_4.fetchall(), columns=result_4.keys())
    df_result_5 = pd.DataFrame(result_5.fetchall(), columns=result_5.keys())

In [56]:
df_result_1

,Regionname,total_registros
0,Eastern Metropolitan,1413
1,Eastern Victoria,49
2,Northern Metropolitan,3229
3,Northern Victoria,36
4,South-Eastern Metropolitan,436
5,Southern Metropolitan,3571
6,Western Metropolitan,2657
7,Western Victoria,32


In [58]:
df_result_5

,Suburb,promedio_precio
0,Kooyong,3.080000e+06
1,Canterbury,2.294347e+06
2,Malvern,2.287286e+06
3,Middle Park,2.225935e+06
4,East Melbourne,2.223889e+06


## Ejercicio 2 - Pandas:

Este ejercicio usa el archivo `airbnb_price_by_zipcode.csv` generado en el notebook `02.1 Combinación de datasets.ipynb`. Si no lo tenés, generarlo primero antes de comenzar esta parte.

1. Seleccionar un subconjunto de columnas que les parezcan relevantes al problema de predicción del valor de la propiedad. Justificar explicitamente las columnas seleccionadas y las que no lo fueron.
  1. Valores faltantes: ¿Qué porcentaje de filas tienen al menos un valor faltante?
  2. Mostrar la dispersión o distribución de las columnas seleccionadas.
 3. Eliminar los valores extremos que no sean relevantes para la predicción de valores de las propiedades.
 4. Mostrar visualmente los valores extremos que eliminás


2. Agregar información adicional respectiva al entorno de una propiedad a partir del [conjunto de datos de AirBnB](https://www.kaggle.com/tylerx/melbourne-airbnb-open-data?select=cleansed_listings_dec18.csv) utilizado en el práctico.
  1. Seleccionar qué variables agregar y qué combinaciones aplicar a cada una. Por ejemplo, pueden utilizar solo la columna `price`, o aplicar múltiples transformaciones como la mediana (¿por qué no la media?) o el mínimo.
  2. Utilizar la variable zipcode para unir los conjuntos de datos. Sólo incluir los zipcodes que tengan una cantidad mínima de registros (a elección) como para que la información agregada sea relevante.
  3. Mostrar un gráfico zipcode vs airbnb_price_median.
  4. Investigar al menos otras 2 variables que puedan servir para combinar los datos, y justificar si serían adecuadas o no. Pueden asumir que cuentan con la ayuda de anotadores expertos para encontrar equivalencias entre barrios o direcciones, o que cuentan con algoritmos para encontrar las n ubicaciones más cercanas a una propiedad a partir de sus coordenadas geográficas. **NO** es necesario que realicen la implementación. Si tuvieras que entrevistar a un experto inmobiliario para mapear barrios entre datasets, ¿qué 3 preguntas le harías para validar esa correspondencia?
  5. Si las coordenadas geoespaciales estuvieran disponibles, como las usarian?

Pueden leer otras columnas del conjunto de AirBnB además de las que están en `interesting_cols`, si les parecen relevantes.

¿Qué cosas no están en los datos que te gustaría tener para predecir mejor el precio de una propiedad?


In [60]:
airbnb_price_by_zipcode.columns

Index(['zipcode', 'airbnb_price_mean', 'airbnb_record_count',
       'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'],
      dtype='object')

In [61]:
airbnb_price_by_zipcode.head()

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,2010.0,40.000000,1,NaN,NaN
1,2134.0,50.000000,1,NaN,NaN
2,2582.0,104.000000,1,NaN,NaN
3,3000.0,150.504307,3367,918.738956,3407.204651
4,3001.0,132.500000,2,NaN,NaN


In [62]:
airbnb_price_by_zipcode.describe()

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
count,247.000000,247.000000,247.000000,184.000000,168.000000
mean,3508.538462,149.900412,92.101215,756.381199,2804.327037
std,1903.914390,84.872988,261.914701,393.460560,1642.559997
min,2010.000000,37.000000,1.000000,133.000000,527.000000
25%,3072.500000,93.744681,8.500000,463.721429,1645.783333
50%,3148.000000,126.012987,27.000000,667.900000,2359.000000
75%,3589.500000,187.168478,73.000000,994.500000,3450.363636
max,30122.000000,759.083333,3367.000000,2236.666667,10060.000000


### Criterios de evaluación
Se evaluará principalmente:
- claridad del código,
- justificación de las decisiones de curación,
- coherencia entre el análisis realizado y las conclusiones,
- presencia de validaciones después de operaciones críticas como merges o cargas a base.

No se espera una única solución correcta, pero sí que las decisiones estén justificadas y sean consistentes con los datos.


## Ejercicio 3:

Crear y guardar un nuevo conjunto de datos con todas las transformaciones realizadas anteriormente.

## Ejercicios opcionales:

El notebook `02.2 ETLs-DAGs.ipynb` tiene un esqueleto de referencia para guiarse.

1. Armar un script en python (archivo .py) [ETL](https://towardsdatascience.com/what-to-log-from-python-etl-pipelines-9e0cfe29950e) que corra los pasos de extraccion, transformacion y carga, armando una funcion para cada etapa del proceso y luego un main que corra todos los pasos requeridos.

2. Armar un DAG en Apache Airflow que corra el ETL. (https://airflow.apache.org/docs/apache-airflow/stable/tutorial.html)

3. Bonus: embeddings y búsqueda semántica con descripciones de AirBnB.
   - Usar `sentence-transformers` para codificar descripciones textuales de propiedades.
   - Tomar un subconjunto chico de descripciones, calcular embeddings y encontrar el par más similar con similitud coseno.
   - Reflexionar: ¿por qué este resultado no se puede lograr con `LIKE '%keyword%'` en SQL? ¿Qué pasa si dos propiedades son similares pero usan palabras distintas? ¿Qué representan los 384 números del embedding?

4. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?


5. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?

Ejemplo conceptual:


In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Valores únicos con posibles inconsistencias
council_values = melb_df['CouncilArea'].dropna().unique().tolist()

message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": f"""Estos son los valores únicos de la columna CouncilArea en un dataset de propiedades de Melbourne:
{council_values}

Identificá: (1) duplicados con distinta capitalización o spelling,
 (2) valores que parecen errores, (3) valores que podrían agruparse.
Respondé en JSON con la estructura: {{"estandarizado": {{"valor_original": "valor_correcto"}}}}"""
    }]
)

mapping = json.loads(message.content[0].text)
melb_df['CouncilArea_clean'] = melb_df['CouncilArea'].map(
    mapping.get('estandarizado', {})
).fillna(melb_df['CouncilArea'])
